# 58 — LSEG/Gemma hybrid uncertainty-aware capacity

**Question.** Does the unchanged Notebook 36 Gemma-timing/FinBERT-ranking hybrid retain a defensible capital envelope after adding sampling uncertainty to Notebook 51's point-estimate implementation curve?

The design was frozen in `frozen_specs/lseg_gemma_hybrid_capacity_uncertainty_v1.json` before bootstrap outcomes were computed. It keeps the original 33 companies, one-session holding period, 25% name cap, 10 bps per traded side, and Notebook 51's primary square-root impact coefficient $Y=1$. The full Notebook 51 AUM grid is evaluated with identical five-session circular-block resamples.

A grid point passes only if the 95% interval for mean daily net return is strictly positive, at least 95% of bootstrapped terminal returns are positive, both observed half-sample net Sharpes are positive, and every order stays below 5% of lagged full-day ADV. The result is an opened-window implementation diagnostic, not new alpha evidence or permission to change the prospective contract. No licensed headline text is loaded or emitted.

In [ ]:
from __future__ import annotations

import hashlib
import json
import subprocess
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == 'final_experiments':
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from final_experiments.lib.capacity import (  # noqa: E402
    ImpactConfig,
    build_lagged_liquidity,
    fold_terminal_liquidation,
    simulate_square_root_impact,
    summarize_capacity,
)
from final_experiments.lib.evaluate import annualized_sharpe  # noqa: E402
from final_experiments.lib.plots import CATEGORICAL, INK, apply_house_style  # noqa: E402
from final_experiments.lib.sector_portfolios import SECTOR_MEMBERS  # noqa: E402
from final_experiments.lib.sparse_spread import build_exact_extrema_targets  # noqa: E402
from sentiment_benchmark.strategy_research.market import OpenToOpenReturn  # noqa: E402
from sentiment_benchmark.strategy_research.portfolio import (  # noqa: E402
    PositionTarget,
    TargetPortfolio,
)

SPEC_PATH = REPO_ROOT / 'final_experiments/frozen_specs/lseg_gemma_hybrid_capacity_uncertainty_v1.json'
N51_DIR = REPO_ROOT / 'final_experiments/outputs/51_lseg_gemma_hybrid_implementation_capacity'
AGGREGATE_DIR = REPO_ROOT / 'final_experiments/outputs/13_lseg_44_gemma_robustness'
GEMMA_PATH = AGGREGATE_DIR / 'gemma4_26b_firm_open_aggregators.parquet'
FINBERT_PATH = AGGREGATE_DIR / 'finbert_firm_open_aggregators.parquet'
PRICE_PATH = REPO_ROOT / 'Data/derived/prices/lseg_us_sector_33_8m.csv'
OUTPUT_DIR = REPO_ROOT / 'final_experiments/outputs/58_lseg_gemma_hybrid_capacity_uncertainty'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

spec = json.loads(SPEC_PATH.read_text())
assert spec['status'] == 'frozen_before_bootstrap_outcomes'
SEED = int(spec['bootstrap']['seed'])
BLOCK_LENGTH = int(spec['bootstrap']['block_length_sessions'])
REPLICATIONS = int(spec['bootstrap']['replications'])
AUM_GRID = tuple(float(value) for value in spec['initial_aum_usd_grid'])
COST_BPS = float(spec['candidate']['fixed_cost_bps_per_side'])
IMPACT_COEFFICIENT = float(spec['candidate']['impact_coefficient_Y'])
PARTICIPATION_LIMIT = float(spec['candidate']['hard_participation_ceiling'])
SINGLE_NAME_CAP = float(spec['candidate']['single_name_cap'])
apply_house_style()
pd.set_option('display.max_columns', 80)
pd.set_option('display.float_format', lambda value: f'{value:,.6f}')

## Exact candidate reconstruction and Notebook 51 identity

Only the aggregate firm-open scorer panels and market data needed to reproduce the frozen candidate are loaded. Every AUM point must reproduce Notebook 51's existing point estimates before uncertainty is calculated.

In [ ]:
sector_members = {sector: tuple(members[:3]) for sector, members in SECTOR_MEMBERS.items()}
original_symbols = tuple(symbol for members in sector_members.values() for symbol in members)
gemma = pd.read_parquet(GEMMA_PATH)
finbert = pd.read_parquet(FINBERT_PATH)
for frame in (gemma, finbert):
    frame['session_date'] = pd.to_datetime(frame['session_date']).dt.normalize()
gemma = gemma.loc[gemma['symbol'].isin(original_symbols)].copy()
finbert = finbert.loc[finbert['symbol'].isin(original_symbols)].copy()
if not gemma[['session_date', 'symbol']].equals(finbert[['session_date', 'symbol']]):
    raise ValueError('paired scorer panels are not row-aligned')
if gemma.duplicated(['session_date', 'symbol']).any() or finbert['strongest_event'].isna().any():
    raise ValueError('candidate scorer input is invalid')

prices = pd.read_csv(PRICE_PATH, parse_dates=['session_date'])
prices['session_date'] = pd.to_datetime(prices['session_date']).dt.normalize()
liquidity = build_lagged_liquidity(
    prices.loc[prices['symbol'].isin(original_symbols)].copy(),
    window_sessions=20,
    minimum_observations=5,
)
open_wide = (
    prices.loc[prices['symbol'].isin(original_symbols)]
    .pivot(index='session_date', columns='symbol', values='open')
    .sort_index()
    .dropna()
)
sessions = pd.DatetimeIndex(sorted(gemma['session_date'].unique()))
symbols = tuple(sorted(gemma['symbol'].unique()))
locations = open_wide.index.get_indexer(sessions)
if len(sessions) != 167 or set(symbols) != set(original_symbols):
    raise ValueError('expected 167 sessions and immutable original 33')
if (locations < 0).any() or not np.array_equal(locations[1:], locations[:-1] + 1):
    raise ValueError('signal sessions are not consecutive complete-price sessions')

def rank_map(frame: pd.DataFrame) -> dict[pd.Timestamp, tuple[str, ...]]:
    result = {
        pd.Timestamp(session): tuple(
            day.sort_values(['strongest_event', 'symbol'], ascending=[False, True], kind='mergesort')['symbol']
        )
        for session, day in frame.groupby('session_date', sort=True)
    }
    if any(set(row) != set(symbols) for row in result.values()):
        raise ValueError('rank map is incomplete')
    return result

_, target_audit = build_exact_extrema_targets(gemma, single_name_cap=SINGLE_NAME_CAP)
ranks = rank_map(finbert)
targets: list[TargetPortfolio] = []
for row in target_audit.itertuples(index=False):
    n_long, n_short, eligible = int(row.positive_names), int(row.negative_names), bool(row.eligible)
    ranked = ranks[pd.Timestamp(row.session_date)]
    long_symbols = set(ranked[:n_long]) if eligible else set()
    short_symbols = set(ranked[-n_short:]) if eligible else set()
    if eligible and (not n_long or not n_short or long_symbols & short_symbols):
        raise RuntimeError('invalid hybrid selection')
    leg = min(0.5, SINGLE_NAME_CAP * n_long, SINGLE_NAME_CAP * n_short) if eligible else 0.0
    weights = {
        symbol: (leg / n_long if symbol in long_symbols else -leg / n_short if symbol in short_symbols else 0.0)
        for symbol in symbols
    }
    vector = np.asarray([weights[symbol] for symbol in symbols], dtype=float)
    if abs(vector.sum()) > 1e-12 or np.abs(vector).max() > SINGLE_NAME_CAP + 1e-12:
        raise RuntimeError('target violates exposure constraints')
    positions = tuple(
        PositionTarget(
            symbol=symbol, action=float(np.sign(weights[symbol])), volatility=None,
            raw_weight=weights[symbol], target_weight=weights[symbol], exclusion_reason=None,
        )
        for symbol in symbols
    )
    targets.append(TargetPortfolio(
        session=str(pd.Timestamp(row.session_date).date()), positions=positions,
        gross_exposure=float(np.abs(vector).sum()), net_exposure=float(vector.sum()),
        long_exposure=float(vector[vector > 0].sum()),
        short_exposure=float(-vector[vector < 0].sum()), cash_weight=1.0 - float(vector.sum()),
    ))

return_rows: list[OpenToOpenReturn] = []
for session, location in zip(sessions, locations, strict=True):
    next_session = open_wide.index[location + 1]
    values = open_wide.loc[next_session, list(symbols)] / open_wide.loc[session, list(symbols)] - 1.0
    return_rows.extend(
        OpenToOpenReturn(symbol=symbol, session=str(session.date()), next_session=str(next_session.date()), value=float(value))
        for symbol, value in values.items()
    )

n51_curve = pd.read_csv(N51_DIR / 'capacity_curve.csv')
n51_primary = n51_curve.loc[n51_curve['impact_coefficient'].eq(IMPACT_COEFFICIENT)].set_index('initial_aum_usd')
simulations = {}
identity_rows = []
for initial_aum in AUM_GRID:
    simulation = simulate_square_root_impact(
        tuple(targets), return_rows, liquidity,
        config=ImpactConfig(
            fixed_cost_rate_per_side=COST_BPS / 10_000.0,
            impact_coefficient=IMPACT_COEFFICIENT,
            initial_nav_usd=initial_aum,
            participation_limit=PARTICIPATION_LIMIT,
        ),
    )
    simulations[initial_aum] = simulation
    metrics = summarize_capacity(simulation, participation_limit=PARTICIPATION_LIMIT)
    reference = n51_primary.loc[initial_aum]
    for metric in ('sharpe_net', 'total_return_net', 'maximum_drawdown', 'maximum_participation'):
        identity_rows.append({
            'initial_aum_usd': initial_aum, 'metric': metric,
            'absolute_error': abs(float(metrics[metric]) - float(reference[metric])),
        })
identity = pd.DataFrame(identity_rows)
if identity['absolute_error'].max() > 1e-12:
    raise RuntimeError('Notebook 51 identity failed')
display(pd.DataFrame([{
    'sessions': len(sessions), 'companies': len(symbols),
    'active_sessions': int(target_audit['eligible'].sum()),
    'aum_grid_points': len(AUM_GRID),
    'maximum_notebook51_identity_error': float(identity['absolute_error'].max()),
    'licensed_headline_text_loaded': False, 'source_regimes_pooled': False,
}]).T)

## Paired block-bootstrap uncertainty frontier

Identical circular-block indices are applied at every AUM so changes across the frontier reflect the implementation model rather than Monte Carlo noise. The bootstrap is conditional on each realised impact-adjusted daily path; it does not re-estimate impact or simulate a new order book.

In [ ]:
def circular_block_indices(n: int, block_length: int, replications: int, seed: int) -> np.ndarray:
    rng = np.random.default_rng(seed)
    blocks_needed = int(np.ceil(n / block_length))
    starts = rng.integers(0, n, size=(replications, blocks_needed))
    offsets = np.arange(block_length)
    return ((starts[..., None] + offsets) % n).reshape(replications, -1)[:, :n]

def sharpe(values: np.ndarray, axis: int = -1) -> np.ndarray:
    mean = np.mean(values, axis=axis)
    std = np.std(values, axis=axis, ddof=1)
    return np.divide(mean * np.sqrt(252.0), std, out=np.zeros_like(mean, dtype=float), where=std > 0)

indices = circular_block_indices(len(sessions), BLOCK_LENGTH, REPLICATIONS, SEED)
frontier_rows = []
for initial_aum in AUM_GRID:
    simulation = simulations[initial_aum]
    daily = fold_terminal_liquidation(simulation.daily).reset_index(drop=True)
    net = daily['net_return'].to_numpy(dtype=float)
    sampled = net[indices]
    boot_means = sampled.mean(axis=1)
    boot_sharpes = sharpe(sampled, axis=1)
    boot_terminal = np.expm1(np.log1p(sampled).sum(axis=1))
    split = (len(net) + 1) // 2
    point = summarize_capacity(simulation, participation_limit=PARTICIPATION_LIMIT)
    mean_low, mean_high = np.quantile(boot_means, [0.025, 0.975])
    sharpe_low, sharpe_high = np.quantile(boot_sharpes, [0.025, 0.975])
    terminal_low, terminal_high = np.quantile(boot_terminal, [0.025, 0.975])
    probability_positive = float(np.mean(boot_terminal > 0))
    first_half_sharpe = float(annualized_sharpe(pd.Series(net[:split])))
    second_half_sharpe = float(annualized_sharpe(pd.Series(net[split:])))
    gate = bool(
        mean_low > 0
        and probability_positive >= 0.95
        and first_half_sharpe > 0
        and second_half_sharpe > 0
        and bool(point['all_orders_within_participation_limit'])
    )
    frontier_rows.append({
        'initial_aum_usd': initial_aum,
        'n_sessions': len(net),
        'active_sessions': int(point['active_sessions']),
        'point_net_sharpe': float(point['sharpe_net']),
        'bootstrap_sharpe_ci_low': float(sharpe_low),
        'bootstrap_sharpe_ci_high': float(sharpe_high),
        'point_mean_net_bps_session': float(net.mean() * 10_000),
        'bootstrap_mean_ci_low_bps_session': float(mean_low * 10_000),
        'bootstrap_mean_ci_high_bps_session': float(mean_high * 10_000),
        'point_total_net_return': float(point['total_return_net']),
        'bootstrap_terminal_ci_low': float(terminal_low),
        'bootstrap_terminal_ci_high': float(terminal_high),
        'bootstrap_probability_terminal_positive': probability_positive,
        'first_half_net_sharpe': first_half_sharpe,
        'second_half_net_sharpe': second_half_sharpe,
        'maximum_participation': float(point['maximum_participation']),
        'orders_over_participation_limit': int(point['orders_over_participation_limit']),
        'all_orders_within_participation_limit': bool(point['all_orders_within_participation_limit']),
        'uncertainty_aware_capacity_gate': gate,
    })
frontier = pd.DataFrame(frontier_rows)
passing = frontier.loc[frontier['uncertainty_aware_capacity_gate'], 'initial_aum_usd']
capacity_aum = float(passing.max()) if len(passing) else float('nan')
display(frontier)
display(pd.DataFrame([{'uncertainty_aware_capacity_aum_usd': capacity_aum, 'passing_grid_points': int(len(passing))}]).T)

## Results and decision

The four panels distinguish expected economics, sampling uncertainty, temporal stability, and the hard liquidity ceiling. A green point passes the complete predeclared implementation gate; it does not become validated alpha.

In [ ]:
x = frontier['initial_aum_usd'].to_numpy(dtype=float) / 1_000_000
colors = np.where(frontier['uncertainty_aware_capacity_gate'], '#59a14f', '#e15759')
fig, axes = plt.subplots(2, 2, figsize=(14.5, 9.2), sharex=True)
axes[0, 0].plot(x, frontier['point_net_sharpe'], color=CATEGORICAL[0], marker='o')
axes[0, 0].fill_between(x, frontier['bootstrap_sharpe_ci_low'], frontier['bootstrap_sharpe_ci_high'], color=CATEGORICAL[0], alpha=0.18)
axes[0, 0].axhline(0, color=INK['reference'], lw=1)
axes[0, 0].set_ylabel('Net Sharpe')
axes[0, 0].set_title('Point estimate and 95% block-bootstrap interval')
axes[0, 1].plot(x, frontier['point_mean_net_bps_session'], color=CATEGORICAL[1], marker='o')
axes[0, 1].fill_between(x, frontier['bootstrap_mean_ci_low_bps_session'], frontier['bootstrap_mean_ci_high_bps_session'], color=CATEGORICAL[1], alpha=0.18)
axes[0, 1].axhline(0, color=INK['reference'], lw=1)
axes[0, 1].set_ylabel('Mean net return (bps/session)')
axes[0, 1].set_title('Cash-relative uncertainty drives the gate')
axes[1, 0].plot(x, frontier['bootstrap_probability_terminal_positive'], color=CATEGORICAL[2], marker='o', label='P(terminal return > 0)')
axes[1, 0].plot(x, frontier['first_half_net_sharpe'], color=CATEGORICAL[3], marker='s', label='First-half Sharpe')
axes[1, 0].plot(x, frontier['second_half_net_sharpe'], color=CATEGORICAL[4], marker='^', label='Second-half Sharpe')
axes[1, 0].axhline(0.95, color=INK['reference'], ls='--', lw=1, label='95% probability gate')
axes[1, 0].axhline(0, color=INK['reference'], lw=0.8)
axes[1, 0].set_ylabel('Probability / Sharpe')
axes[1, 0].set_title('Bootstrap confidence and temporal stability')
axes[1, 0].legend(frameon=False, fontsize=8)
axes[1, 1].plot(x, 100 * frontier['maximum_participation'], color=CATEGORICAL[5], marker='o')
axes[1, 1].scatter(x, 100 * frontier['maximum_participation'], c=colors, s=65, zorder=3)
axes[1, 1].axhline(100 * PARTICIPATION_LIMIT, color=INK['reference'], ls='--', lw=1, label='5% ADV ceiling')
axes[1, 1].set_ylabel('Maximum order / lagged ADV (%)')
axes[1, 1].set_title('Hard order-participation constraint')
axes[1, 1].legend(frameon=False, fontsize=8)
for axis in axes.flat:
    axis.set_xscale('log')
    axis.grid(True, which='major', alpha=0.2)
for axis in axes[1, :]:
    axis.set_xlabel('Initial AUM (USD millions, log scale)')
fig.suptitle('Frozen LSEG/Gemma-FinBERT hybrid: uncertainty-aware implementation capacity')
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'capacity_uncertainty_frontier.png', dpi=180, bbox_inches='tight')
plt.show()

frontier.to_csv(OUTPUT_DIR / 'uncertainty_frontier.csv', index=False)
identity.to_csv(OUTPUT_DIR / 'notebook51_identity.csv', index=False)
decision = {
    'uncertainty_aware_capacity_aum_usd': None if np.isnan(capacity_aum) else capacity_aum,
    'passing_grid_points': int(len(passing)),
    'point_estimate_capacity_aum_usd_notebook51': float(n51_primary.loc[n51_primary['capacity_gate_at_grid_point']].index.max()),
    'prospective_candidate_changed': False,
    'validated_alpha': False,
    'deployment_qualified': False,
}
try:
    git_commit = subprocess.run(['git', 'rev-parse', 'HEAD'], cwd=REPO_ROOT, check=True, capture_output=True, text=True).stdout.strip()
except (OSError, subprocess.CalledProcessError):
    git_commit = None
manifest = {
    'status': 'RETROSPECTIVE_OPENED_WINDOW_UNCERTAINTY_AWARE_CAPACITY_DIAGNOSTIC',
    'notebook': '58_lseg_gemma_hybrid_capacity_uncertainty.ipynb',
    'git_commit_at_execution': git_commit,
    'frozen_spec_sha256': hashlib.sha256(SPEC_PATH.read_bytes()).hexdigest(),
    'specification': spec,
    'input_boundary': {
        'gemma_aggregate_panel': str(GEMMA_PATH.relative_to(REPO_ROOT)),
        'finbert_aggregate_panel': str(FINBERT_PATH.relative_to(REPO_ROOT)),
        'price_path': str(PRICE_PATH.relative_to(REPO_ROOT)),
        'notebook51_capacity_curve': str((N51_DIR / 'capacity_curve.csv').relative_to(REPO_ROOT)),
        'licensed_headline_text_loaded': False, 'licensed_text_emitted': False,
        'source_regimes_pooled': False, 'row_level_returns_persisted': False,
    },
    'identity': {'maximum_error_vs_notebook51': float(identity['absolute_error'].max())},
    'decision': decision,
    'frontier': json.loads(frontier.to_json(orient='records')),
    'claim_boundary': {
        'new_strategy_constructed': False, 'strategy_reselected': False,
        'historical_retuning_performed': False, 'prospective_contract_changed': False,
        'alpha_promotion_permitted': False,
    },
    'limitations': [
        'The candidate and historical return window were selected before this diagnostic.',
        'Bootstrap uncertainty is conditional on the realised impact-adjusted daily paths and does not simulate new order books.',
        'Full-day ADV is optimistic for an opening-auction strategy.',
        'The square-root impact coefficient is a scenario assumption rather than an estimate from this corpus.',
        'Borrow, financing, taxes, short-sale constraints, and order-book depth remain unmodelled.',
        'Human validation of full-corpus Gemma sentiment and genuinely new-date replay remain required.',
    ],
}
(OUTPUT_DIR / 'manifest.json').write_text(json.dumps(manifest, indent=2) + '\n')

display(Markdown(
    '### Decision\n\n'
    f"- Notebook 51's point-estimate capacity was **${decision['point_estimate_capacity_aum_usd_notebook51'] / 1_000_000:.0f}m**.\n"
    + (f"- The stricter uncertainty-aware capacity is **${capacity_aum / 1_000_000:.0f}m**.\n" if np.isfinite(capacity_aum) else '- **No predeclared AUM grid point clears the uncertainty-aware gate.**\n')
    + '- This does not validate alpha or change the frozen prospective strategy. The result only bounds how much capital the retrospective path can support under the declared implementation assumptions.'
))
print(json.dumps(decision, indent=2))

## Next steps

1. Preserve the full frontier, including a zero-capacity result if no grid point passes.
2. Do not alter the candidate, impact coefficient, block length, AUM grid, or gate after seeing these outcomes.
3. Carry the exact Notebook 36 primary into the frozen prospective replay once its population gates pass.
4. Re-estimate capacity only on genuinely new data or with independently sourced execution-cost evidence.